In [1]:
import pandas as pd
import numpy as np
import os

import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

##Loading in Dataset

1. Curl getting and unzipping the dataset from kaggle
2. Loading all the csvs into pandas so we can figure out what we want and not etc.
3. We will start with the `olist_orders_dataset` and progressively merge other related datasets to create a comprehensive `working_data` DataFrame.

#### Download and merge dataset

In [3]:
#!/bin/bash
!curl -L -o /content/brazilian-ecommerce.zip https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce
!unzip /content/brazilian-ecommerce.zip -d /content/brazilian-ecommerce
!ls /content/brazilian-ecommerce

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 42.6M    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (56) Failure writing output to destination, passed 1207 returned 4294967295
unzip:  cannot find or open /content/brazilian-ecommerce.zip, /content/brazilian-ecommerce.zip.zip or /content/brazilian-ecommerce.zip.ZIP.
ls: /content/brazilian-ecommerce: No such file or directory


Let's define the base path for the unzipped dataset and then load each CSV file into a pandas DataFrame.

In [11]:
dataset_path = './content/brazilian-ecommerce'

# List of CSV files to load
csv_files = [
    'olist_customers_dataset.csv',
    'olist_geolocation_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'product_category_name_translation.csv'
]

dfs = []

for file in csv_files:
    file_path = os.path.join(dataset_path, file)
    df_name = file.replace('.csv', '') # Create a clean name for the DataFrame variable
    # print(f"Loading {file}...")
    # Load the CSV into a DataFrame with a dynamic variable name
    globals()[df_name] = pd.read_csv(file_path)
    dfs.append(globals()[df_name])
    # print(f"Successfully loaded {df_name}. Shape: {globals()[df_name].shape}")
    # print(f"First 5 rows of {df_name}:")
    # display(globals()[df_name].head())
    # print("\n---\n") # Separator for readability

In [12]:
# Start with the orders dataset
working_data = olist_orders_dataset.copy()

print(f"Initial working_data shape: {working_data.shape}")

# Merge with order_items (needed for seller + product info)
# Note: An order can have multiple items, so this merge will increase the number of rows if an order has more than one item.
working_data = pd.merge(
    working_data,
    olist_order_items_dataset,
    on='order_id',
    how='left'
)
print(f"Shape after merging with order_items: {working_data.shape}")

# Merge with order_payments (payment behavior)
# Note: An order can have multiple payments, so this merge might also increase the number of rows.
working_data = pd.merge(
    working_data,
    olist_order_payments_dataset,
    on='order_id',
    how='left'
)
print(f"Shape after merging with order_payments: {working_data.shape}")

# Merge with customers (location)
working_data = pd.merge(
    working_data,
    olist_customers_dataset,
    on='customer_id',
    how='left'
)
print(f"Shape after merging with customers: {working_data.shape}")

# Optionally: Merge with products
# This adds product specific details
working_data = pd.merge(
    working_data,
    olist_products_dataset,
    on='product_id',
    how='left'
)
print(f"Shape after merging with products: {working_data.shape}")

# Optionally: Merge with sellers
# This adds seller specific details
working_data = pd.merge(
    working_data,
    olist_sellers_dataset,
    on='seller_id',
    how='left'
)
print(f"Shape after merging with sellers: {working_data.shape}")

# Merge with product_category_name_translation (for English product categories)
product_category_name_translation = pd.concat([product_category_name_translation, pd.DataFrame(
    {'product_category_name': ['portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'],
     'product_category_name_english': ['portable_kitchen_and_food_preparers', 'pc_gamer']}
    )], ignore_index=True)

working_data = pd.merge(
    working_data,
    product_category_name_translation,
    on='product_category_name',
    how='left'
)
print(f"Shape after merging with product_category_name_translation: {working_data.shape}")

# Merge with olist_order_reviews_dataset
working_data = pd.merge(
    working_data,
    olist_order_reviews_dataset,
    on='order_id',
    how='left'
)
print(f"Shape after merging with product_category_name_translation: {working_data.shape}")

# Display the first 5 rows and the full shape of the combined DataFrame
# print("\nFirst 5 rows of the combined working_data:")
# display(working_data.head())
# print(f"\nFinal working_data shape: {working_data.shape}")

Initial working_data shape: (99441, 8)
Shape after merging with order_items: (113425, 14)
Shape after merging with order_payments: (118434, 18)
Shape after merging with customers: (118434, 22)
Shape after merging with products: (118434, 30)
Shape after merging with sellers: (118434, 33)
Shape after merging with product_category_name_translation: (118434, 34)
Shape after merging with product_category_name_translation: (119143, 40)


#### Deal with the dataset

In [13]:
DATE_COLS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date", "shipping_limit_date", "review_creation_date", "review_answer_timestamp"]
for col in DATE_COLS:
  working_data[col] = pd.to_datetime(working_data[col])

In [14]:

# Only looking at delivered orders with delivery dates
working_data = working_data.dropna(subset=["order_delivered_customer_date"]).copy()
working_data = working_data[working_data["order_status"].isin(["delivered"])].copy()

# Delete unnecessary columns
# DROP_COLS = [
#     "review_id", "review_score", "review_comment_title",
#     "review_comment_message", "review_creation_date",
#     "review_answer_timestamp", "order_delivered_carrier_date"
# ]
# working_data = working_data.drop(columns=DROP_COLS)

# Fill missing values in datetime cols
working_data["order_approved_at"] = working_data["order_approved_at"].fillna(working_data["order_purchase_timestamp"])

# Fix payment columns
working_data["payment_type"] = working_data["payment_type"].fillna(working_data["payment_type"].mode()[0])
working_data["payment_installments"] = working_data["payment_installments"].fillna(1)
working_data["payment_value"] = working_data["payment_value"].fillna(working_data["payment_value"].median())
working_data["payment_sequential"] = working_data["payment_sequential"].fillna(1)

# Impute numeric values like name/description length or weight/dimension or photos etc. on median
product_num_cols = [
    "product_name_lenght", "product_description_lenght",
    "product_photos_qty", "product_weight_g",
    "product_length_cm", "product_height_cm", "product_width_cm"
]

for col in product_num_cols:
    working_data[col] = working_data.groupby(
        "product_category_name_english"
    )[col].transform(lambda x: x.fillna(x.median()))

    # fallback if still NA
    working_data[col] = working_data[col].fillna(
        working_data[col].median()
    )

# Delete rows with missing values
working_data = working_data.dropna(subset=["product_category_name"]).copy()

## Feature Creation

#### Order status

In [15]:
working_data['is_delayed'] = (working_data['order_delivered_customer_date'] > working_data['order_estimated_delivery_date']).astype(int)
working_data['is_canceled'] = (working_data['order_status'] == 'canceled').astype(int)

#### Distance

max_distance = max(haversine_distance(customer lat/long, seller lat/long)) // Latitude/Longitude joined on zipcode prefix

// Considering if avg or min would matter?

In [16]:
# Distance between Seller & Customer
ageoloc = olist_geolocation_dataset.groupby('geolocation_zip_code_prefix')\
    [['geolocation_lat', 'geolocation_lng']].mean().reset_index()

# Prepare customer geolocation data by renaming columns before merging
customer_geoloc = ageoloc.rename(columns={
    'geolocation_lat': 'customer_lat',
    'geolocation_lng': 'customer_lng',
    'geolocation_zip_code_prefix': 'customer_zip_code_prefix_geo' # Temporary column for merge key
})

# Merge with geolocation data for customer's location
working_data = pd.merge(
    working_data,
    customer_geoloc,
    left_on='customer_zip_code_prefix',
    right_on='customer_zip_code_prefix_geo',
    how='left'
)
# Drop the temporary merge key column
working_data.drop(columns=['customer_zip_code_prefix_geo'], axis=1, errors='ignore', inplace=True)

# Prepare seller geolocation data by renaming columns before merging
seller_geoloc = ageoloc.rename(columns={
    'geolocation_lat': 'seller_lat',
    'geolocation_lng': 'seller_lng',
    'geolocation_zip_code_prefix': 'seller_zip_code_prefix_geo' # Temporary column for merge key
})

# Merge with geolocation data for seller's location
working_data = pd.merge(
    working_data,
    seller_geoloc,
    left_on='seller_zip_code_prefix',
    right_on='seller_zip_code_prefix_geo',
    how='left'
)
# Drop the temporary merge key column
working_data.drop(columns=['seller_zip_code_prefix_geo'], axis=1, errors='ignore', inplace=True)

# Define Haversine distance function
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in kilometers

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    distance = R * c
    return distance

# Calculate distance between customer and seller locations
working_data['distance'] = working_data.apply(
    lambda row: haversine_distance(
        row['customer_lat'], row['customer_lng'],
        row['seller_lat'], row['seller_lng']
    ) if pd.notna(row['customer_lat']) and pd.notna(row['customer_lng']) and \
         pd.notna(row['seller_lat']) and pd.notna(row['seller_lng'])
      else np.nan, # Assign NaN if any coordinate is missing
    axis=1
)

print("\nFirst 5 rows of working_data after geolocation merge and distance calculation:")
display(working_data.head())


First 5 rows of working_data after geolocation merge and distance calculation:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,review_comment_message,review_creation_date,review_answer_timestamp,is_delayed,is_canceled,customer_lat,customer_lng,seller_lat,seller_lng,distance
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,0,0,-23.576983,-46.587161,-23.680729,-46.444238,18.576110
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,0,0,-23.576983,-46.587161,-23.680729,-46.444238,18.576110
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,0,0,-23.576983,-46.587161,-23.680729,-46.444238,18.576110
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,...,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,0,0,-12.177924,-44.660711,-19.807681,-43.980427,851.495069
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,aa4383b373c6aca5d8797843e5594415,...,NaN,2018-08-18,2018-08-22 19:07:58,0,0,-16.745150,-48.514783,-21.363502,-48.229601,514.410666


#### Product Volume

product_volume = product_length_cm * product_height_cm * product_width_cm

In [17]:
working_data['product_volume'] = working_data['product_length_cm'] * working_data['product_height_cm'] * working_data['product_width_cm']

#### Function to create stats till point

In [18]:
def add_asof_rate(
    df,
    group_col,
    event_time_col,
    prediction_time_col,
    outcome_col,
    new_col,
    default_value=0
):
    df = df.copy()

    history = (
        df[[group_col, event_time_col, outcome_col]]
        .dropna(subset=[group_col, event_time_col])
        .sort_values([event_time_col, group_col])
        .copy()
    )

    history['_cum_count'] = history.groupby(group_col).cumcount() + 1
    history['_cum_sum'] = history.groupby(group_col)[outcome_col].cumsum()
    history['_rate'] = history['_cum_sum'] / history['_cum_count']

    history = history[[group_col, event_time_col, '_rate']].sort_values(
        [group_col, event_time_col]
    )

    left = (
    df[[group_col, prediction_time_col]]
    .reset_index()
    .rename(columns={'index': '_orig_index'})
    .sort_values([prediction_time_col, group_col])
    )

    history = history.sort_values([event_time_col, group_col])

    out = pd.merge_asof(
        left,
        history,
        left_on=prediction_time_col,
        right_on=event_time_col,
        by=group_col,
        direction='backward',
        allow_exact_matches=False
    )

    df[new_col] = out.set_index('_orig_index')['_rate']
    df[new_col] = df[new_col].fillna(default_value)

    return df

#### Seller Info

In [19]:
working_data = add_asof_rate(
    working_data,
    group_col='seller_id',
    event_time_col='order_delivered_customer_date',
    prediction_time_col='order_purchase_timestamp',
    outcome_col='is_delayed',
    new_col='delay_rate',
    default_value=0
)

working_data = add_asof_rate(
    working_data,
    group_col='seller_id',
    event_time_col='order_delivered_customer_date',
    prediction_time_col='order_purchase_timestamp',
    outcome_col='is_canceled',
    new_col='cancel_rate',
    default_value=0
)

#### Product Category

In [20]:
working_data = add_asof_rate(
    working_data,
    group_col='product_category_name_english',
    event_time_col='order_delivered_customer_date',
    prediction_time_col='order_purchase_timestamp',
    outcome_col='is_delayed',
    new_col='product_delay_rate',
    default_value=0
)

working_data = add_asof_rate(
    working_data,
    group_col='product_category_name_english',
    event_time_col='order_delivered_customer_date',
    prediction_time_col='order_purchase_timestamp',
    outcome_col='is_canceled',
    new_col='product_cancel_rate',
    default_value=0
)

#### Date Variables

In [21]:
working_data['purchase_hour'] = working_data['order_purchase_timestamp'].dt.hour
working_data['purchase_dayofweek'] = working_data['order_purchase_timestamp'].dt.dayofweek

working_data['approval_delay_hours'] = (working_data['order_approved_at'] - working_data['order_purchase_timestamp']).dt.total_seconds() / 3600

#### y Variables


In [22]:
working_data['delivery_days'] = (working_data['order_delivered_customer_date'] - working_data['order_purchase_timestamp']).dt.total_seconds() / (60 * 60 * 24)
working_data['delivery_days_estimated'] = (working_data['order_estimated_delivery_date'] - working_data['order_purchase_timestamp']).dt.total_seconds() / (60 * 60 * 24)

#### Filter out data

In [23]:
working_data["delay_days"] = np.maximum(0, working_data['delivery_days']-working_data['delivery_days_estimated']).round(0).astype(int)
working_data["early_days"] = np.maximum(0, working_data['delivery_days_estimated']-working_data['delivery_days']).round(0).astype(int)


In [25]:
import statsmodels.api as sm

# --------------------------------------------------
# Review-based cost calibration
# --------------------------------------------------

# Assumes working_data has:
# review_score
# delay_days: 0 if not delayed, positive if late
# early_days: 0 if not early, positive if early

review_cols = ['review_score', 'delay_days', 'early_days']

review_data = working_data[review_cols].dropna().copy()

# Optional: cap extreme values so 100+ day outliers do not dominate
cap_days = 30
review_data['delay_days_capped'] = review_data['delay_days'].clip(upper=cap_days)
review_data['early_days_capped'] = review_data['early_days'].clip(upper=cap_days)

# --------------------------------------------------
# Model 1: review impact of delay
# --------------------------------------------------
X_delay = sm.add_constant(review_data[['delay_days_capped']])
y = review_data['review_score']

delay_model = sm.OLS(y, X_delay).fit(cov_type='HC1')

# --------------------------------------------------
# Model 2: review impact of early delivery
# --------------------------------------------------
X_early = sm.add_constant(review_data[['early_days_capped']])

early_model = sm.OLS(y, X_early).fit(cov_type='HC1')

# --------------------------------------------------
# Extract coefficients
# --------------------------------------------------
delay_effect = delay_model.params['delay_days_capped']
early_effect = early_model.params['early_days_capped']

# Delay effect is negative, early effect is positive
review_based_ratio = abs(delay_effect) / abs(early_effect)

print("Delay coefficient:", delay_effect)
print("Early coefficient:", early_effect)
print("Review-based late/conservative cost ratio:", review_based_ratio)

# Convert to usable loss-function weights
late_cost = round(review_based_ratio)
conservative_cost = 1

print(f"\nRecommended review-based costs:")
print(f"late_cost = {late_cost}")
print(f"conservative_cost = {conservative_cost}")
print(f"Ratio = {late_cost}:1")

Delay coefficient: -0.13913647895019762
Early coefficient: 0.028643565663500993
Review-based late/conservative cost ratio: 4.8575125242697

Recommended review-based costs:
late_cost = 5
conservative_cost = 1
Ratio = 5:1


In [26]:
missing_data = working_data.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if not missing_data.empty:
    missing_percentage = (missing_data / len(working_data)) * 100
    missing_info = pd.DataFrame({
        'Missing Count': missing_data,
        'Missing Percentage (%)': missing_percentage
    })
    print("Columns with Missing Values in working_data (sorted by percentage):")
    display(missing_info)
else:
    print("No missing values found in working_data.")

Columns with Missing Values in working_data (sorted by percentage):


,Missing Count,Missing Percentage (%)
review_comment_title,100575,88.164135
review_comment_message,66704,58.472786
review_id,849,0.744234
review_score,849,0.744234
review_creation_date,849,0.744234
review_answer_timestamp,849,0.744234
distance,557,0.488267
customer_lat,298,0.261227
customer_lng,298,0.261227
seller_lat,260,0.227916


In [27]:
print(len(working_data))
working_data = working_data.dropna(subset=['delivery_days'])
print(len(working_data))

114077
114077


#### Create final feature dataset

1. `purchase_hour`: Time that the order was put in.
3. `purchase_dayofweek`: Day that the order was put in.
4. `approval_delay_hours`: Time between order put in to accepted. This is not data leakage.

1. `customer_unique_id`: Unique customer information.
4. `customer_city`: City of customer delivery // NOT BEING USED
5. `customer_state`: State of customer delivery // NOT BEING USED

6. `max_distance`: Maximum distance any of the items in the order are travelling
7. `num_items`: Number of items in order
8. `order_price`: Sum of price of all products in order
9. `shipping_cost`: Sum of freight_value of all products in order

10. `order_weight`: Sum of weight of all products in order
11. `order_volume`: Sum of volume of all products in order

12. `avg_photos_per_product`: Average of photos of all products in order
13. `avg_product_description_length`: Average of description length of all products in order

14. `n_states`: Number of unique seller states in an order.
15. `n_cities`: Number of unique seller cities in an order.

16. `avg_delay_sellers`: Average delay rate of sellers for an order.
17. `median_delay_sellers`: Median delay rate of sellers for an order.
18. `min_delay_sellers`: Minimum delay rate of sellers for an order.
19. `max_delay_sellers`: Maximum delay rate of sellers for an order.
20. `p25_delay_sellers`: 25th percentile delay rate of sellers for an order.
21. `p75_delay_sellers`: 75th percentile delay rate of sellers for an order.

22. `avg_cancel_sellers`: Average cancellation rate of sellers for an order.
23. `mediancancel_sellers`: Median cancellation rate of sellers for an order.
24. `min_cancel_sellers`: Minimum cancellation rate of sellers for an order.
25. `max_cancel_sellers`: Maximum cancellation rate of sellers for an order.
26. `p25_cancel_sellers`: 25th percentile cancellation rate of sellers for an order.
27. `p75_cancel_sellers`: 75th percentile cancellation rate of sellers for an order.

28. `avg_delay_product`: Average delay rate of product category for an order.
29. `median_delay_product`: Median delay rate of product category for an order.
30. `min_delay_product`: Minimum delay rate of product category for an order.
31. `max_delay_product`: Maximum delay rate of product category for an order.
32. `p25_delay_product`: 25th percentile delay rate of product category for an order.
33. `p75_delay_product`: 75th percentile delay rate of product category for an order.

34. `avg_cancel_product`: Average cancellation rate of product category for an order.
35. `mediancancel_product`: Median cancellation rate of product category for an order.
36. `min_cancel_product`: Minimum cancellation rate of product category for an order.
37. `max_cancel_product`: Maximum cancellation rate of product category for an order.
38. `p25_cancel_product`: 25th percentile cancellation rate of product category for an order.
39. `p75_cancel_product`: 75th percentile cancellation rate of product category for an order.

y-values
1. `delivery_days`: Number of days to delivery -- TRUTH
2. `delivery_days_estimated`: Number of days estimated to delivery -- BASELINE
4. `order_status`: Current status of order.


In [28]:
final_df = (
    working_data.groupby('order_id', as_index=False)
    .agg(
        customer_unique_id=('customer_unique_id', 'first'),
        order_status=('order_status', 'first'),
        order_purchase_timestamp=('order_purchase_timestamp', 'first'),
        purchase_hour=('purchase_hour', 'first'),
        purchase_dayofweek=('purchase_dayofweek', 'first'),
        # approval_delay_hours=('approval_delay_hours', 'first'),
        # order_delivered_carrier_date=('order_delivered_carrier_date', 'first'),
        delivery_days=('delivery_days', 'first'),
        delivery_days_estimated=('delivery_days_estimated', 'first'),
        customer_city=('customer_city', 'first'),
        customer_state=('customer_state', 'first'),
        max_distance=('distance', 'max'),
        num_items=('product_id', 'count'),
        order_price=('price', 'sum'),
        # shipping_cost=('freight_value', 'sum'),
        order_weight=('product_weight_g', 'sum'),
        order_volume=('product_volume', 'sum'),
        avg_photos_per_product=('product_photos_qty', 'mean'),
        avg_product_description_length=('product_description_lenght', 'mean'),
        n_states=('seller_state', 'nunique'),
        n_cities=('seller_city', 'nunique'),

        # Seller level delays & cancellations
        avg_delay_sellers=('delay_rate', 'mean'),
        median_delay_sellers=('delay_rate', 'median'),
        min_delay_sellers=('delay_rate', 'min'),
        max_delay_sellers=('delay_rate', 'max'),
        p25_delay_sellers=('delay_rate', lambda x: x.quantile(0.25)),
        p75_delay_sellers=('delay_rate', lambda x: x.quantile(0.75)),
        avg_cancel_sellers=('cancel_rate', 'mean'),
        mediancancel_sellers=('cancel_rate', 'median'),
        min_cancel_sellers=('cancel_rate', 'min'),
        max_cancel_sellers=('cancel_rate', 'max'),
        p25_cancel_sellers=('cancel_rate', lambda x: x.quantile(0.25)),
        p75_cancel_sellers=('cancel_rate', lambda x: x.quantile(0.75)),

        avg_delay_product=('product_delay_rate', 'mean'),
        median_delay_product=('product_delay_rate', 'median'),
        min_delay_product=('product_delay_rate', 'min'),
        max_delay_product=('product_delay_rate', 'max'),
        p25_delay_product=('product_delay_rate', lambda x: x.quantile(0.25)),
        p75_delay_product=('product_delay_rate', lambda x: x.quantile(0.75)),
        avg_cancel_product=('product_cancel_rate', 'mean'),
        mediancancel_product=('product_cancel_rate', 'median'),
        min_cancel_product=('product_cancel_rate', 'min'),
        max_cancel_product=('product_cancel_rate', 'max'),
        p25_cancel_product=('product_cancel_rate', lambda x: x.quantile(0.25)),
        p75_cancel_product=('product_cancel_rate', lambda x: x.quantile(0.75)),

    )
)
display(final_df.head())

,order_id,customer_unique_id,order_status,order_purchase_timestamp,purchase_hour,purchase_dayofweek,delivery_days,delivery_days_estimated,customer_city,customer_state,...,min_delay_product,max_delay_product,p25_delay_product,p75_delay_product,avg_cancel_product,mediancancel_product,min_cancel_product,max_cancel_product,p25_cancel_product,p75_cancel_product
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,delivered,2017-09-13 08:59:02,8,2,7.614421,15.625671,campos dos goytacazes,RJ,...,0.032895,0.032895,0.032895,0.032895,0.0,0.0,0.0,0.0,0.0,0.0
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,delivered,2017-04-26 10:53:06,10,2,16.216181,18.546458,santa fe do sul,SP,...,0.025424,0.025424,0.025424,0.025424,0.0,0.0,0.0,0.0,0.0,0.0
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,delivered,2018-01-14 14:33:31,14,6,7.948437,21.393391,para de minas,MG,...,0.070655,0.070655,0.070655,0.070655,0.0,0.0,0.0,0.0,0.0,0.0
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,delivered,2018-08-08 10:00:35,10,2,6.147269,11.582928,atibaia,SP,...,0.074413,0.074413,0.074413,0.074413,0.0,0.0,0.0,0.0,0.0,0.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,delivered,2017-02-04 13:57:51,13,5,25.114352,40.418160,varzea paulista,SP,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
missing_data = final_df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if not missing_data.empty:
    missing_percentage = (missing_data / len(final_df)) * 100
    missing_info = pd.DataFrame({
        'Missing Count': missing_data,
        'Missing Percentage (%)': missing_percentage
    })
    print("Columns with Missing Values in final_df (sorted by percentage):")
    display(missing_info)
else:
    print("No missing values found in final_df.")

Columns with Missing Values in final_df (sorted by percentage):


,Missing Count,Missing Percentage (%)
max_distance,470,0.494019


In [30]:
final_df["max_distance"] = final_df["max_distance"].fillna(final_df["max_distance"].median())

In [31]:
final_df.to_csv('final_df_20260503.csv', index=False)

In [32]:
working_data.to_csv('working_data_20260503.csv', index=False)